# MIRI MRS SNR maps with forward modeling

This notebook computes a detection SNR map around a star from MIRI MRS stage 2 `cal` files, using the BREADS forward model (`compute_snr_grid_miri`):

1. **Data**: select the stage 2 `cal` files of one target and band (as produced by `demo_mrs_calibration.ipynb`).
2. **Stellar spectrum**: combined, continuum-normalized stellar spectrum of the sequence.
3. **Recentering** (optional): fit of the star position in each exposure with an STPSF model.
4. **SNR map**: planet forward model fitted on a grid of RA/Dec offsets.

## Environment

The thread limits must be set **before** numpy is imported (first cell of the notebook): the forward model runs in a pool of processes, and multi-threaded BLAS in each of them would oversubscribe the machine.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["CRDS_PATH"] = os.path.expanduser("~/crds_cache")
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"
# STPSF data files, needed for use_stpsf=True. STPSF_PATH must point to an existing directory
stpsf_path = os.path.expanduser("~/data/stpsf-data")
if os.path.isdir(stpsf_path):
    os.environ["STPSF_PATH"] = stpsf_path

In [ ]:
import multiprocessing as mp

import numpy as np

import breads.jwst_tools.default_miri as default
from breads.jwst_reduction.reduction_utils import (
    retrieve_mrs_data,
    compute_normalized_stellar_spectrum_miri,
    recenter_coordinates_of_sequence_miri,
    compute_snr_grid_miri,
)
from breads.grid_search import limit_blas_threads

## Configuration

Change those before running the notebook.

In [ ]:
numthreads = 10  # number of processes of the pool

uncaldir = "/path/to/uncal/"  # same root directory as in demo_mrs_calibration.ipynb
target_name = "* eps Ind Ab"  # target name as in the TARGNAME header keyword (name of the reduction subdirectory)
simbad_name = "eps indi"  # name used to query the star in SIMBAD

channel = "2"  # Select the desired MRS channel to run the forward modeling
channel_band = "2A"  # channel + sub-band to analyze
combined_channel_band = "12A"  # combined channel band of the reduction directory (both channels of the detector)

photfilter_filename = "/path/to/JWST_MIRI.F560W.dat"  # photometric filter used to normalize the companion model
model_grid_h5py = "/path/to/atmospheric_model_grid.h5"  # grid of companion atmospheric models
teff, logg, vsini = 250, 4.0, 0  # companion model parameters
grid_paras = (teff, logg, vsini)

# Grid of RA/Dec offsets (arcsec) where the SNR is computed
ra_vec_snr = np.arange(-2.5, 2.5, 0.05)
dec_vec_snr = np.arange(-2.5, 2.5, 0.05)
star_rv = 0  # km/s
suffix = "snr_map"

## Data and output directories

In [ ]:
def create_fm_saving_paths(uncaldir, target_name, channel_band, N_nodes):
    utils_dir = os.path.join(uncaldir, target_name, f"utils_fm_snr_{N_nodes}_nodes", channel_band)
    print(utils_dir)
    os.makedirs(utils_dir, exist_ok=True)
    out_dir = os.path.join(uncaldir, target_name, f"fm_snr_outputs_{N_nodes}_nodes", channel_band)
    print(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    return utils_dir, out_dir

x_nodes = default.nodes_dict[channel_band]
N_nodes = len(x_nodes)
print(f"Number of nodes:{N_nodes}")
print(f"Nodes spacing used :{x_nodes} micron")
utils_dir, out_dir = create_fm_saving_paths(uncaldir, target_name, channel_band, N_nodes)
filelist = retrieve_mrs_data(uncaldir, target_name, combined_channel_band)


splitbasename = os.path.basename(filelist[0]).split("_")
# One BLAS thread per worker, otherwise the workers oversubscribe the CPU
mypool = mp.Pool(processes=numthreads, initializer=limit_blas_threads)

## Combined stellar spectrum

In [ ]:
combined_contnorm_spec_filename = os.path.join(
    utils_dir, splitbasename[0] + "_" + splitbasename[1] + "_starspec_contnorm_combined_1dspline.fits")
compute_normalized_stellar_spectrum_miri(filelist, channel, utils_dir, combined_contnorm_spec_filename,
                                         mppool=mypool, targetname=simbad_name, overwrite=False)

## Recentering (optional)

Fits the star position in each exposure with an STPSF model. Set `run_recentering = True` to run it.
This is still in development especially for channel 2. Skip for now.

In [ ]:
run_recentering = False

fitpsf_stpsf_filename = os.path.join(utils_dir, splitbasename[0] + "_fitspsf_stpsf.fits")
if run_recentering:
    poly_centroid_filename, poly_fluxcal_filename = recenter_coordinates_of_sequence_miri(
        filelist, channel, utils_dir, combined_contnorm_spec_filename,
        fitpsf_stpsf_filename, use_stpsf=True,
        mppool=mypool, IWA=1.5, OWA=3,
        overwrite=False,
        targetname=simbad_name,
        stis_spectrum=None)

## SNR map

This wrapper function runs the full forward modeling for MRS.
- it normalizes the planet template flux in the desired photometric filter. This is useful to directly have both the flux and the flux error estimates in coherent physical units for the contrast curves.
- it calls grid_search function to apply the forward modeling for each point of the RV, RA, DEC grid.

The function will save the outputs in the "snr_map" subdirectory of utils_dir

In [ ]:
compute_snr_grid_miri(filelist, channel, utils_dir, os.path.join(out_dir, "snr_map"),
                      combined_contnorm_spec_filename,
                      None,
                      model_grid_h5py,
                      grid_paras,
                      photfilter_filename,
                      aper_radius=0.25, #aperture radius used to fit each spatial location of the gid
                      suffix=suffix, contnorm_suffix=None,
                      x_vec=ra_vec_snr, y_vec=dec_vec_snr, rv_vec=np.array([star_rv]), ## Grid parameters for the forward model
                      ra_dec_point_sources=None, ## optional: RA/DEC list of companion position to mask when estimating the star spectrum
                      mppool=mypool,
                      overwrite=False,
                      targetname=simbad_name,
                      use_stpsf=True, use_breadspsf=False)

In [ ]:
mypool.close()
mypool.join()